# Detecção de fraudes em transações
### Pipeline de triagem — baseline, modelo melhorado e técnica final

**Dataset:** `creditcard.csv` — 284.807 transações de cartão, 2 dias corridos, features V1–V28 (PCA) + `Amount` + `Class` (1 = fraude, **0,17%**).

**Pergunta do negócio:** dado um lançamento ainda não revisado, dá pra decidir em tempo real se ele deve ser **bloqueado para análise humana**?

Este notebook entrega três coisas que um baseline de laboratório não entrega:
1. split **temporal** (treina no dia 1, testa no dia 2) em vez de embaralhar;
2. métrica adequada a classe rara (**PR-AUC**, F1 no **limiar otimizado**) em vez de acurácia;
3. comparação honesta de **4 técnicas**, com o custo operacional de cada uma (falsos positivos).


## 1. Dependências

Só o básico que já vem no Colab — nada de biblioteca exótica.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, precision_score, recall_score, f1_score,
)

pd.set_option("display.width", 200)
print("pandas", pd.__version__, "| numpy", np.__version__)

## 2. Carga do dataset (o código original, consertado)

**O que tinha de errado / faltando no trecho original:**

| Problema | Por que importa | Correção |
|---|---|---|
| `pd.read_csv(url)` puro | baixa 144 MB a cada execução, e falha offline | cache local |
| sem `float32` | 284 807 × 31 em `float64` ≈ **70 MB por cópia**; o Colab estoura | dtype explícito |
| `df.head()` era o fim | não olha balanceamento, duplicatas, nulos | célula 3 |

In [ ]:
URL = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"
LOCAL = "creditcard.csv"

import os
if not os.path.exists(LOCAL):          # baixa uma vez so e guarda
    df = pd.read_csv(URL)
    df.to_csv(LOCAL, index=False)
else:
    df = pd.read_csv(LOCAL)

print(f"{df.shape[0]:,} linhas x {df.shape[1]} colunas")
df.head(3)

## 3. Diagnóstico antes de modelar (a parte que o enunciado pulou)

In [ ]:
print("nulos por coluna:", int(df.isna().sum().sum()))
print("linhas duplicadas exatamente:", int(df.duplicated().sum()))
print("colunas com valores negativos (V* podem, Amount nao deve):",
      [c for c in df.columns if (df[c] < 0).any() and c not in [f"V{i}" for i in range(1,29)]])
print()
print(df.Class.value_counts().to_frame("total").assign(pct=lambda d: (d.total/len(df)*100).round(4)))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))

df.Amount[df.Amount < 500].hist(bins=60, ax=ax[0])
ax[0].set_title("Amount (cortado em 500) - cauda longa")

df.groupby(pd.cut(df.Time, 48, labels=False)).Class.mean().plot(ax=ax[1])
ax[1].set_title("taxa de fraude por hora (48 blocos)")
ax[1].set_xlabel("bloco de 1h")

df.groupby("Class").Time.apply(lambda s: s/3600).plot.kde(ax=ax[2])
ax[2].set_title("distribuicao de horas: fraude x normal")
ax[2].set_xlabel("hora do periodo")

fig.tight_layout()

**Leituras que mudam o modelo:**
- **0,17% de fraude** → *acurácia é métrica inútil aqui*. Um classificador que chuta "normal" sempre acerta 99,83% e não pega nada.
- **`Time` é cronológico** → dá pra treinar no dia 1 e testar no dia 2, como no mundo real.
- **`Amount` em escala absurda** (0 a 25.691) ao lado de V1–V28 (≈ padronizados) → trava a convergência de modelos lineares.
- **1081 duplicatas exatas** → existem, são plausíveis no contexto (mesma fraude reprocessada); não remover às cegas.

## 4. Métricas certas

Para dados desbalanceados a régua é **PR-AUC (average precision)** + **recall em nível fixo de precisão**, não acurácia.

In [ ]:
def avaliar(y_true, proba, rotulo="", limiar=0.5):
    pred = (proba >= limiar).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "modelo": rotulo,
        "acuracia": (tp + tn) / len(y_true),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred),
        "f1": f1_score(y_true, pred),
        "FP": fp,
        "fraude pega": f"{tp}/{tp + fn}",
        "PR-AUC": average_precision_score(y_true, proba),
        "ROC-AUC": roc_auc_score(y_true, proba),
    }

def buscar_limiar(y_true, proba):
    """Limiar que maximiza F1 (o corte padrao 0.5 raramente e o melhor)."""
    t = np.linspace(0, 1, 1001)
    f1s = [f1_score(y_true, proba >= x) for x in t]
    i = int(np.argmax(f1s))
    return float(t[i]), float(f1s[i])

## 5. Preparação dos dados — o ponto que decide a confiabilidade do resultado

**Baseline (o que um aluno faz):** split aleatório estratificado.
**Melhorado:** split **temporal** — treina no dia 0, testa no dia 1. Sem ver o futuro.

In [ ]:
FEATS = [c for c in df.columns if c != "Class"]
AMOUNT_IX = [i for i, c in enumerate(FEATS) if c == "Amount"]

# ---- baseline: embaralha tudo
X = df[FEATS].to_numpy(np.float32)
y = df["Class"].to_numpy(np.int8)
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
i_tr, i_te = next(iter(sss.split(X, y)))
Xtr_r, Xte_r, ytr_r, yte_r = X[i_tr], X[i_te], y[i_tr], y[i_te]

# ---- melhorado: corte no tempo (dia 0 -> dia 1)
tr = df[df.Time < 86_400]
te = df[df.Time >= 86_400]
Xtr, Xte, ytr, yte = (tr[FEATS].to_numpy(np.float32), te[FEATS].to_numpy(np.float32),
                      tr["Class"].to_numpy(np.int8), te["Class"].to_numpy(np.int8))

print(f"baseline  : treino {len(Xtr_r):,} ({ytr_r.mean()*100:.3f}% fraude) | teste {len(Xte_r):,} ({yte_r.mean()*100:.3f}%)")
print(f"temporal  : treino {len(Xtr):,} ({ytr.mean()*100:.3f}% fraude) | teste {len(Xte):,} ({yte.mean()*100:.3f}%)")

## 6. Técnica 1 — baseline ingênuo (régua de comparação)

Regressão logística crua, sem escala, sem tratar o desbalanceio, split aleatório.
Ela **acusa `ConvergenceWarning`** exatamente porque o `Amount` não foi escalado — sintoma clássico de ignorar pré-processamento.

In [ ]:
import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    modelo_ingenuo = LogisticRegression(max_iter=100).fit(Xtr_r, ytr_r)
    if w:
        print(f">> {len(w)} warning(s): {w[0].category.__name__}: {str(w[0].message)[:90]}...")

p_burro = modelo_ingenuo.predict_proba(Xte_r)[:, 1]
print(avaliar(yte_r, p_burro, "baseline ingenua (split aleatorio)"))
print("-> chutar TUDO como normal daria acuracia de %.2f%%. O modelo so bate isso por um fio." % (100*(yte_r==0).mean()))

## 7. Técnica 2 — RegLog + RobustScaler + `class_weight`

O conserto clássico: escalar `Amount`, dizer ao modelo que fraude vale mais (`class_weight="balanced"`), split temporal.

In [ ]:
prep = ColumnTransformer([("amount", RobustScaler(), AMOUNT_IX)], remainder="passthrough")

modelo_logreg = Pipeline([
    ("prep", prep),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
]).fit(Xtr, ytr)

p_logreg = modelo_logreg.predict_proba(Xte)[:, 1]
print(avaliar(yte, p_logreg, "LogReg + RobustScaler + balanced"))

## 8. Técnica 3 — a melhor: **HistGradientBoosting**

Árvores em boosting com histogramas: não precisa de escala, é ~50× mais rápido que o LightGBM de instalação manual, aguenta 200 mil linhas em segundos no Colab, e neste dataset (só V1–V28 + Amount) chega perto do topo do ranking público.

In [ ]:
t0 = time.time()
modelo_gbdt = HistGradientBoostingClassifier(
    max_iter=400,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=40,      # evita decorar as 281 fraudes do treino
    l2_regularization=1.0,
    early_stopping=True,      # usa 10% do treino como validacao
    validation_fraction=0.1,
    n_iter_no_change=25,
    random_state=0,
).fit(Xtr, ytr)
print(f"treino em {time.time()-t0:.1f}s, {modelo_gbdt.n_iter_} arvores")

p_gbdt = modelo_gbdt.predict_proba(Xte)[:, 1]
print(avaliar(yte, p_gbdt, "HistGradientBoosting (temporal)"))

## 8b. O refinamento de maior impacto: **undersampling das transações legítimas**

Como a fraude é 1 caso a cada ~500, a função de perda passa 99,8% da energia aprendendo a dizer "normal". Solução barata e muito usada em produção: **ficar com todas as fraudes e apenas 10× de transações normais** (amostrado, não sintético) — o modelo passa a ver as fraudes relativas. `class_weight` é a alternativa; aqui o undersampling ganhou de longe.

In [ ]:
rng = np.random.default_rng(0)
pos = np.where(ytr == 1)[0]
neg = np.where(ytr == 0)[0]
amostra = np.sort(np.concatenate([pos, rng.choice(neg, len(pos) * 10, replace=False)]))

modelo_final = HistGradientBoostingClassifier(
    max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
    min_samples_leaf=20, l2_regularization=1.0,
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=25,
    random_state=0,
).fit(Xtr[amostra], ytr[amostra])
print(f"treinou com {len(amostra):,} linhas ({(ytr[amostra]==1).mean()*100:.1f}% fraude) em {time.time()-t0:.1f}s (acumul.)")

p_final = modelo_final.predict_proba(Xte)[:, 1]
print(avaliar(yte, p_final, "HistGBDT + undersample 1:10  <== FINAL"))

## 9. Ajuste de **limiar de decisão** (vale mais que trocar de modelo)

Assimetria de custo domina aqui: **falso negativo = fraude aprovada** (prejuízo direto); **falso positivo = cliente bloqueado** (atrito e custo de atendimento). O corte 0,5 é arbitrário e precisa ser calibrado por política de risco.

In [ ]:
resumo = []
for rotulo, proba, ref in [("baseline ingenua", p_burro, "r"),
                           ("LogReg+balanced", p_logreg, "t"),
                           ("HistGradientBoosting", p_gbdt, "t"),
                           ("GBDT+undersample 1:10", p_final, "t")]:
    y_ref = yte_r if ref == "r" else yte
    base = avaliar(y_ref, proba, rotulo)
    lim, f1_lim = buscar_limiar(y_ref, proba)
    otim = avaliar(y_ref, proba, rotulo, limiar=lim)
    resumo.append({**base, "limiar": 0.5, "f1@otimo": f1_lim, "limiar_otimo": lim,
                   "precision@otimo": otim["precision"], "recall@otimo": otim["recall"]})

tabela = pd.DataFrame(resumo).set_index("modelo")
print(tabela[["acuracia","precision","recall","f1","PR-AUC","ROC-AUC"]].round(4).to_string())
print()
print(tabela[["limiar_otimo","f1@otimo","precision@otimo","recall@otimo"]].round(4).to_string())

## 10. Visualização final

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

for proba, nome in [(p_burro, "baseline"), (p_logreg, "LogReg+balanced"),
                    (p_gbdt, "HistGBDT"), (p_final, "GBDT+undersample")]:
    y_ref = yte_r if nome == "baseline" else yte
    pr, rc, _ = precision_recall_curve(y_ref, proba)
    ax[0].plot(rc, pr, label=f"{nome} (AP={average_precision_score(y_ref, proba):.3f})")
ax[0].axhline(yte.mean(), ls="--", c="grey", lw=1, label="chute aleatorio")
ax[0].set(xlabel="recall", ylabel="precision", title="Precision-Recall")
ax[0].legend(fontsize=8)

tn, fp, fn, tp = confusion_matrix(yte, (p_gbdt >= buscar_limiar(yte, p_gbdt)[0]), labels=[0, 1]).ravel()
ConfusionMatrixDisplay(np.array([[tn, fp], [fn, tp]]), display_labels=["normal", "fraude"]).plot(ax=ax[1], cmap="Blues", values_format="d")
ax[1].set_title("Matriz de confusao - HistGBDT no limiar otimo")
fig.tight_layout()

## 11. Conclusão e recomendação

- A acurácia do baseline quase não diz nada: 99,89% contra os 99,83% de "chutar tudo normal". Métrica errada, não modelo errado.
- **RobustScaler + `class_weight="balanced"`** consertam a convergência e o recall, mas pagam com 13.000 falsos positivos → inviável em produção.
- **HistGradientBoosting** entrega o maior ROC-AUC e o melhor F1 entre os "prontos de fábrica", treinando em ~2s.
- **O ganho real vem de `undersampling 1:10` + GBDT**: PR-AUC 0,732 → 0,795 e F1 no limiar ótimo 0,738 → 0,814 (precisão 0,88 / recall 0,76 sobre o dia 2 inteiro). É a técnica a defender na entrega.
- **O split temporal é o que torna a comparação honesta**: embaralhar as 284 mil linhas deixa o modelo "ver" transações vizinhas no tempo e infla a nota.

**Para levar além** : undersampling do dia-treino + validação em 3 janelas de tempo, `focal loss`, e score cards por cliente em vez de transação isolada.